# 📊 QLoRA Fine-Tuning Project — Notebook 2: Baseline Evaluation
**Goal:** Evaluate Mistral-7B on CNN/DailyMail using 3 prompting strategies (no fine-tuning)  

## Strategies Evaluated
1. **Zero-Shot** — Basic instruction, no examples
2. **Few-Shot** — 3 examples in context
3. **Optimized Prompt** — Carefully engineered prompt with structure

## What This Notebook Does
- Loads the processed dataset from Notebook 1
- Runs inference with Mistral-7B in 4-bit quantization (no training)
- Evaluates all 3 strategies on 1000 test samples
- Saves predictions and ROUGE scores for Notebook 4

**Runtime:** ~3-4 hours  
**Input:** `qlora-cnn-processed` dataset  
**Output:** `baseline_results.csv`, `baseline_predictions.csv`

In [1]:
# ══════════════════════════════════════════════════════
# 📦 CELL 1: Install Dependencies
# ══════════════════════════════════════════════════════
# Install newer bitsandbytes that doesn't use broken triton.ops
!pip install -q bitsandbytes>=0.45.0 --upgrade
!pip install -q transformers==4.40.0
!pip install -q datasets==2.18.0
!pip install -q accelerate==0.28.0
!pip install -q peft==0.10.0
!pip install -q evaluate==0.4.1
!pip install -q rouge-score==0.1.2
!pip install -q sentencepiece==0.2.0

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Verify bitsandbytes works
import importlib, bitsandbytes
importlib.reload(bitsandbytes)
print(f"✅ bitsandbytes: {bitsandbytes.__version__}")
print("✅ All dependencies ready")

✅ bitsandbytes: 0.49.2
✅ All dependencies ready


In [2]:
# ══════════════════════════════════════════════════════
# ⚙️ CELL 2: Config & Paths
# ══════════════════════════════════════════════════════
import os, json, numpy as np, pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────
INPUT_DATASET = "/kaggle/input/datasets/pranitsaundankar/qlora-cnn-processed/qlora_project/processed_dataset"
BASE_PATH     = "/kaggle/working/qlora_project"
RESULTS_DIR   = f"{BASE_PATH}/results"
DATA_DIR      = f"{BASE_PATH}/data"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ── Config ─────────────────────────────────────────────
MODEL_ID        = "mistralai/Mistral-7B-v0.1"
MAX_INPUT_LEN   = 512
MAX_NEW_TOKENS  = 128
BATCH_SIZE      = 4   # for inference (no gradient, so larger batch ok)

print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"✅ Config ready")

🔥 GPU: Tesla T4
   VRAM: 15.6 GB
✅ Config ready


## 📥 Load Dataset

In [3]:
# ══════════════════════════════════════════════════════
# 📥 CELL 3: Load Processed Dataset
# ══════════════════════════════════════════════════════
from datasets import load_from_disk

print(f"📥 Loading dataset from {INPUT_DATASET}...")
dataset = load_from_disk(INPUT_DATASET)

test_dataset  = dataset["test"]
train_dataset = dataset["train"]  # needed for few-shot examples

print(f"✅ Dataset loaded")
print(f"   Test  : {len(test_dataset):,}")
print(f"   Train : {len(train_dataset):,}")
print(f"   Columns: {test_dataset.column_names}")

# Preview
print(f"\n🔍 Sample test example:")
print(f"   Input  : {test_dataset[0]['input'][:200]}...")
print(f"   Output : {test_dataset[0]['output']}")

📥 Loading dataset from /kaggle/input/datasets/pranitsaundankar/qlora-cnn-processed/qlora_project/processed_dataset...
✅ Dataset loaded
   Test  : 1,000
   Train : 8,000
   Columns: ['input', 'output']

🔍 Sample test example:
   Input  : The lawyers for a female Pennsylvania police officer charged with criminal homicide last month are trying to keep a video of her fatally shooting an unarmed motorist in the back out of the courtroom. ...
   Output : Pennsylvania police officer Lisa Mearkle charged with criminal homicide .
Stun gun took video of her fatally shooting David Kassick, 59, in the back .
Mearkle, 36, has claimed to authorities the shooting was act of self-defense .
Her attorneys filed motion to prevent DA from showing clip during hearing .
Lawyer for Kassick's family said video 'leaves nothing to the imagination .


## 🤖 Load Mistral-7B (4-bit Quantized)

In [4]:
# ══════════════════════════════════════════════════════
# 🤖 CELL 4: Load Model (4-bit, Inference Only)
# ══════════════════════════════════════════════════════
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"   # left padding for inference

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading model (4-bit)... this takes ~2-3 minutes")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager"   # 👈 add this
)
model.eval()

print(f"\n✅ Model loaded")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading model (4-bit)... this takes ~2-3 minutes


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


✅ Model loaded
   Parameters: 3.8B


## 📝 Define Prompting Strategies

In [5]:
# ══════════════════════════════════════════════════════
# 📝 CELL 5: Define 3 Prompting Strategies
# ══════════════════════════════════════════════════════

# ── Strategy 1: Zero-Shot ──────────────────────────────
def make_zero_shot_prompt(article):
    return f"""Summarize the following news article in 2-3 sentences.

Article:
{article}

Summary:""".strip()

# ── Strategy 2: Few-Shot (3 examples) ─────────────────
FEW_SHOT_EXAMPLES = train_dataset.select(range(3))

def make_few_shot_prompt(article):
    examples = ""
    for i, ex in enumerate(FEW_SHOT_EXAMPLES):
        examples += f"""Example {i+1}:
Article: {ex['input'][:300]}...
Summary: {ex['output']}

"""
    return f"""Summarize news articles concisely. Here are some examples:

{examples}Now summarize this article:
Article: {article}
Summary:""".strip()

# ── Strategy 3: Optimized Prompt ──────────────────────
def make_optimized_prompt(article):
    return f"""<s>[INST] You are a professional news editor. Your task is to write a concise, accurate summary of the following news article.

Instructions:
- Write 2-3 sentences maximum
- Include the most important facts (who, what, when, where)
- Use clear, simple language
- Do not add information not in the article

Article:
{article}

Write a concise summary: [/INST]""".strip()

print("✅ Three prompting strategies defined:")
print("   1. Zero-Shot")
print("   2. Few-Shot (3 examples)")
print("   3. Optimized Prompt")

✅ Three prompting strategies defined:
   1. Zero-Shot
   2. Few-Shot (3 examples)
   3. Optimized Prompt


## ⚡ Inference Functions

In [6]:
# ══════════════════════════════════════════════════════
# ⚡ CELL 6: Batch Inference Function
# ══════════════════════════════════════════════════════
def generate_summaries(prompts, batch_size=BATCH_SIZE):
    all_summaries = []
    
    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        batch_prompts = prompts[i:i+batch_size]
        
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN
        ).to(model.device)
        
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,          # greedy for reproducibility
                temperature=1.0,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True
            )
        
        for j, output in enumerate(outputs):
            input_len = inputs["attention_mask"][j].sum().item()
            generated = tokenizer.decode(
                output[input_len:],
                skip_special_tokens=True
            ).strip()
            # Clean up common artifacts
            generated = generated.split("\n")[0].strip()
            all_summaries.append(generated)
        
        # Clear GPU cache periodically
        if i % (batch_size * 10) == 0:
            torch.cuda.empty_cache()
    
    return all_summaries

print("✅ Inference function ready")

✅ Inference function ready


## 🚀 Run All 3 Strategies

In [7]:
# ══════════════════════════════════════════════════════
# 🚀 CELL 7: Run Zero-Shot Inference
# ══════════════════════════════════════════════════════
print("🔄 Strategy 1: Zero-Shot")
print(f"   Running on {len(test_dataset)} test samples...\n")

zero_shot_prompts = [make_zero_shot_prompt(ex['input']) for ex in test_dataset]
zero_shot_preds   = generate_summaries(zero_shot_prompts)

# Save checkpoint immediately
zs_path = os.path.join(RESULTS_DIR, "zero_shot_predictions.json")
with open(zs_path, "w") as f:
    json.dump(zero_shot_preds, f)
print(f"\n✅ Zero-shot done. Sample: {zero_shot_preds[0][:100]}...")
print(f"   Saved to {zs_path}")

🔄 Strategy 1: Zero-Shot
   Running on 1000 test samples...



Generating:   0%|          | 0/250 [00:00<?, ?it/s]2026-05-12 19:06:32.129635: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778612792.152798     743 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778612792.160348     743 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778612792.197387     743 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778612792.197403     743 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778612792.197405     74

KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════
# 🚀 CELL 8: Run Few-Shot Inference
# ══════════════════════════════════════════════════════
print("🔄 Strategy 2: Few-Shot")
print(f"   Running on {len(test_dataset)} test samples...\n")

few_shot_prompts = [make_few_shot_prompt(ex['input']) for ex in test_dataset]
few_shot_preds   = generate_summaries(few_shot_prompts)

# Save checkpoint immediately
fs_path = os.path.join(RESULTS_DIR, "few_shot_predictions.json")
with open(fs_path, "w") as f:
    json.dump(few_shot_preds, f)
print(f"\n✅ Few-shot done. Sample: {few_shot_preds[0][:100]}...")
print(f"   Saved to {fs_path}")

In [ ]:
# ══════════════════════════════════════════════════════
# 🚀 CELL 9: Run Optimized Prompt Inference
# ══════════════════════════════════════════════════════
print("🔄 Strategy 3: Optimized Prompt")
print(f"   Running on {len(test_dataset)} test samples...\n")

optimized_prompts = [make_optimized_prompt(ex['input']) for ex in test_dataset]
optimized_preds   = generate_summaries(optimized_prompts)

# Save checkpoint immediately
op_path = os.path.join(RESULTS_DIR, "optimized_predictions.json")
with open(op_path, "w") as f:
    json.dump(optimized_preds, f)
print(f"\n✅ Optimized done. Sample: {optimized_preds[0][:100]}...")
print(f"   Saved to {op_path}")

## 📈 Evaluate with ROUGE

In [ ]:
# ══════════════════════════════════════════════════════
# 📈 CELL 10: Compute ROUGE Scores
# ══════════════════════════════════════════════════════
import evaluate

rouge = evaluate.load("rouge")
references = [ex['output'] for ex in test_dataset]

def compute_rouge(predictions, references, name):
    scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_aggregator=True
    )
    print(f"\n📊 {name}:")
    print(f"   ROUGE-1: {scores['rouge1']:.4f}")
    print(f"   ROUGE-2: {scores['rouge2']:.4f}")
    print(f"   ROUGE-L: {scores['rougeL']:.4f}")
    return scores

print("Computing ROUGE scores for all 3 strategies...\n")

zs_scores  = compute_rouge(zero_shot_preds,  references, "Zero-Shot")
fs_scores  = compute_rouge(few_shot_preds,   references, "Few-Shot")
op_scores  = compute_rouge(optimized_preds,  references, "Optimized Prompt")

print("\n✅ All ROUGE scores computed")

In [ ]:
# ══════════════════════════════════════════════════════
# 💾 CELL 11: Save Results to CSV
# ══════════════════════════════════════════════════════

# ── Baseline results summary ───────────────────────────
baseline_results = pd.DataFrame([
    {
        "method": "Zero-Shot",
        "rouge1": zs_scores["rouge1"],
        "rouge2": zs_scores["rouge2"],
        "rougeL": zs_scores["rougeL"],
        "rougeLsum": zs_scores["rougeLsum"]
    },
    {
        "method": "Few-Shot",
        "rouge1": fs_scores["rouge1"],
        "rouge2": fs_scores["rouge2"],
        "rougeL": fs_scores["rougeL"],
        "rougeLsum": fs_scores["rougeLsum"]
    },
    {
        "method": "Optimized Prompt",
        "rouge1": op_scores["rouge1"],
        "rouge2": op_scores["rouge2"],
        "rougeL": op_scores["rougeL"],
        "rougeLsum": op_scores["rougeLsum"]
    }
])

results_path = os.path.join(RESULTS_DIR, "baseline_results.csv")
baseline_results.to_csv(results_path, index=False)
print(f"✅ Saved baseline_results.csv")
print(baseline_results.to_string(index=False))

# ── Predictions CSV ────────────────────────────────────
baseline_preds_df = pd.DataFrame({
    "reference":  references,
    "zero_shot":  zero_shot_preds,
    "few_shot":   few_shot_preds,
    "optimized":  optimized_preds
})

preds_path = os.path.join(RESULTS_DIR, "baseline_predictions.csv")
baseline_preds_df.to_csv(preds_path, index=False)
print(f"✅ Saved baseline_predictions.csv ({len(baseline_preds_df)} rows)")

In [ ]:
# ══════════════════════════════════════════════════════
# 📊 CELL 12: Visualize & Save Baseline Comparison Plot
# ══════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 6))

methods = baseline_results["method"].tolist()
x = np.arange(len(methods))
width = 0.25

bars1 = ax.bar(x - width,   baseline_results["rouge1"], width, label="ROUGE-1", color="steelblue",   alpha=0.85)
bars2 = ax.bar(x,           baseline_results["rouge2"], width, label="ROUGE-2", color="coral",        alpha=0.85)
bars3 = ax.bar(x + width,   baseline_results["rougeL"], width, label="ROUGE-L", color="mediumseagreen", alpha=0.85)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=8)

ax.set_xlabel("Prompting Strategy", fontsize=12)
ax.set_ylabel("ROUGE Score", fontsize=12)
ax.set_title("Baseline Evaluation — Prompting Strategies on CNN/DailyMail", fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, max(baseline_results["rouge1"].max(), 0.5) + 0.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, "baseline_comparison.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved baseline_comparison.png")

In [ ]:
# ══════════════════════════════════════════════════════
# 🔍 CELL 13: Qualitative Examples
# ══════════════════════════════════════════════════════
print("🔍 Qualitative Comparison (5 Examples)\n")
print("=" * 80)

for i in range(5):
    print(f"\n📌 Example {i+1}")
    print(f"ARTICLE    : {test_dataset[i]['input'][:200]}...")
    print(f"REFERENCE  : {references[i]}")
    print(f"ZERO-SHOT  : {zero_shot_preds[i]}")
    print(f"FEW-SHOT   : {few_shot_preds[i]}")
    print(f"OPTIMIZED  : {optimized_preds[i]}")
    print("─" * 80)

## ✅ Notebook 2 Complete!

### Results Summary:
| Strategy | ROUGE-1 | ROUGE-2 | ROUGE-L |
|----------|---------|---------|---------|
| Zero-Shot | — | — | — |
| Few-Shot | — | — | — |
| Optimized Prompt | — | — | — |

### What was saved:
| File | Description |
|------|-------------|
| `baseline_results.csv` | ROUGE scores for all 3 strategies |
| `baseline_predictions.csv` | All predictions + references (1000 rows) |
| `baseline_comparison.png` | Bar chart comparison |
| `zero_shot_predictions.json` | Raw zero-shot predictions |
| `few_shot_predictions.json` | Raw few-shot predictions |
| `optimized_predictions.json` | Raw optimized predictions |

### Next Steps:
1. **Commit this notebook**
2. **Create dataset** `qlora-baseline-results` from output
3. **Run Notebook 3** — QLoRA Fine-tuning